In [2]:
# 1. Import Required Libraries (PySpark)
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, lower, array_distinct, when, lit, collect_set, array_union
import os

spark = SparkSession.builder \
    .appName("NutritionalValuesGenerator") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# In cluster use the HDFS path prefix
path_prefix = "hdfs:///projects/BDA-12/"
# In local use the local path prefix
# path_prefix = "" 

spark.sparkContext.setLogLevel("WARN")

In [3]:
base_dir = f"{path_prefix}converted-dataset"

# 2. Load only necessary Parquet Files as Spark DataFrames
food = spark.read.parquet(f"{base_dir}/food.parquet")
branded_food = spark.read.parquet(f"{base_dir}/branded_food.parquet")
food_attribute = spark.read.parquet(f"{base_dir}/food_attribute.parquet")

In [4]:
# 3. Merge food and branded_food for description and ingredients
food_merged = food.join(branded_food, "fdc_id", "left")

food_merged.show(5)

+-------+------------+--------------------+--------------------+----------------+--------------------+----------+-------------+--------------+--------------------+---------------------------+------------+-----------------+--------------------------+---------------------+-----------+--------------+-------------+--------------+--------------+-----------------+----------------------+-------------+-----------------+-------------+
| fdc_id|   data_type|         description|    food_category_id|publication_date|         brand_owner|brand_name|subbrand_name|      gtin_upc|         ingredients|not_a_significant_source_of|serving_size|serving_size_unit|household_serving_fulltext|branded_food_category|data_source|package_weight|modified_date|available_date|market_country|discontinued_date|preparation_state_code|trade_channel|short_description|material_code|
+-------+------------+--------------------+--------------------+----------------+--------------------+----------+-------------+-------------

In [5]:
# 4. Extract and Normalize Ingredients from branded_food and food_attribute
food_merged = food_merged.withColumn("ingredients_list", split(lower(col("ingredients")), ",|;|CONTAINS:"))

attr_ingredients = food_attribute.filter(col("name") == "Ingredients")
attr_ingredients = attr_ingredients.groupBy("fdc_id").agg(collect_set("value").alias("attr_ingredients_raw"))
attr_ingredients = attr_ingredients.withColumn("attr_ingredients", split(lower(col("attr_ingredients_raw")[0]), ",|;|CONTAINS:"))

food_merged = food_merged.join(attr_ingredients.select("fdc_id", "attr_ingredients"), "fdc_id", "left")

food_merged = food_merged.withColumn(
    "all_ingredients",
    array_distinct(
        array_union(
            when(col("ingredients_list").isNotNull(), col("ingredients_list")).otherwise(array_distinct(lit([]))),
            when(col("attr_ingredients").isNotNull(), col("attr_ingredients")).otherwise(array_distinct(lit([])))
        )
    )
)

In [6]:
# 5. Select only ingredients and product link columns
ingredient_cols = ["fdc_id", "description", "all_ingredients"]
custom_df = food_merged.select(*[col(c) for c in ingredient_cols if c in food_merged.columns])

custom_df.show(5)

+-------+--------------------+--------------------+
| fdc_id|         description|     all_ingredients|
+-------+--------------------+--------------------+
|1105904|WESSON Vegetable ...|  [vegetable oil, 3]|
|1627998|           CHOCOLATE|[sugar,  sunflowe...|
|1628000|      STRAWBERRY JAM|[strawberries,  h...|
|1628002| MAGIC SHELL CUPCAKE|[sugar,  sunflowe...|
|1628003|SUGAR FREE CHERRY...|[water*,  polydex...|
+-------+--------------------+--------------------+
only showing top 5 rows



In [8]:
# 6. Save to Parquet
output_dir = f"{path_prefix}output/ingredients_nutrional_profiles"
if not output_dir.startswith("hdfs://"):
    os.makedirs(output_dir, exist_ok=True)
custom_df.write.mode("overwrite").parquet(output_dir)
print(f"Saved ingredients nutritional profiles to {output_dir}")

Saved ingredients nutritional profiles to hdfs:///projects/BDA-12/output/ingredients_nutrional_profiles
